# 07 Distributions: chances and cut-offs

`sdata.distribution(...)` with `find_chance_that_value` and `find_cut_off`.

**Only simplydata runs in this notebook.** The reference values are in `references/07_distributions.csv`, written by
`scripts/references/07_distributions.py`: first the printed tables of the NIST/SEMATECH e-Handbook of Statistical
Methods (section 1.3.6.7: the standard normal, t, chi-square and F tables), allowing for their printed rounding; then,
on a grid of values the tables do not cover, R (`pnorm`, `qt`, `qchisq`, `qf`, ...), to a relative difference of
1e-12. The same underneath: SciPy (`scipy.stats`). Each check prints its source.

In [1]:
import simplydata as sdata

from checks import ReferenceChecks, quiet, versions

versions("simplydata")
checks = ReferenceChecks("07_distributions")

Python 3.12.16 on Linux 7.0.0-1020-azure
simplydata 0.2.0 (commit 3f0192b of github.com/bioforkgithub/simplydata)
simplydata 0.2.0


R version 4.6.2 (2026-10-29)


## Published table values

- Normal (table 1.3.6.7.1): the area from 0 to 1.96 is 0.47500, so the chance above 1.96 is 0.02500.
- t (table 1.3.6.7.2): the upper 2.5 % point with 10 degrees of freedom is 2.228.
- Chi-square (table 1.3.6.7.4): the upper 5 % point with 10 degrees of freedom is 18.307.
- F (table 1.3.6.7.3): the upper 5 % point with 5 and 10 degrees of freedom is 3.326.

In [2]:
normal = sdata.distribution("normal", average=0, standard_deviation=1)
t10 = sdata.distribution("t", degrees_of_freedom=10)
chi10 = sdata.distribution("chi-square", degrees_of_freedom=10)
f5_10 = sdata.distribution("F", degrees_of_freedom_1=5, degrees_of_freedom_2=10)
checks.check("standard normal table", "find_chance_that_value", "chance above 1.96",
             normal.find_chance_that_value(is_above=1.96))
checks.check("t table", "find_cut_off", "upper 2.5 % point, 10 df", t10.find_cut_off(with_chance_above=0.025))
checks.check("chi-square table", "find_cut_off", "upper 5 % point, 10 df", chi10.find_cut_off(with_chance_above=0.05))
checks.check("F table", "find_cut_off", "upper 5 % point, 5 and 10 df", f5_10.find_cut_off(with_chance_above=0.05))

The chance of a value above 1.96 is 2.5 % (normal, average 0, standard deviation 1; R: pnorm(1.96, mean = 0, sd = 1, lower.tail = FALSE)).
agrees  standard normal table | find_chance_that_value | chance above 1.96: simplydata 0.0249978951482, NIST/SEMATECH e-Handbook 1.3.6.7, printed 0.025; difference 2.10e-06 (allowed 5e-06)
      Source: NIST/SEMATECH e-Handbook 1.3.6.7, printed (printed) - https://www.itl.nist.gov/div898/handbook/eda/section3/eda3671.htm
The cut-off with a 2.5 % chance above it is 2.23 (t, 10 degrees of freedom; R: qt(0.025, df = 10, lower.tail = FALSE)).
agrees  t table | find_cut_off | upper 2.5 % point, 10 df: simplydata 2.22813885199, NIST/SEMATECH e-Handbook 1.3.6.7, printed 2.228; difference 1.39e-04 (allowed 5e-04)
      Source: NIST/SEMATECH e-Handbook 1.3.6.7, printed (printed) - https://www.itl.nist.gov/div898/handbook/eda/section3/eda3672.htm
The cut-off with a 5 % chance above it is 18.3 (chi-square, 10 degrees of freedom; R: qchisq(0.05, df = 10, lower.

1

## A grid of values against R

The chance below each value, and the cut-off with each chance below it.

In [3]:
cases = {
    "normal(10, 2)": (dict(name="normal", average=10, standard_deviation=2), [4.0, 7.5, 10.0, 12.3, 17.0]),
    "t(3)": (dict(name="t", degrees_of_freedom=3), [-6.0, -1.0, 0.0, 0.7, 4.5]),
    "chi-square(4)": (dict(name="chi-square", degrees_of_freedom=4), [0.2, 1.0, 3.36, 9.49, 20.0]),
    "F(3, 12)": (dict(name="F", degrees_of_freedom_1=3, degrees_of_freedom_2=12), [0.1, 0.5, 1.0, 3.49, 8.0]),
}
chances = [0.001, 0.025, 0.5, 0.9, 0.999]
for label, (spec, values) in cases.items():
    spec = dict(spec)
    dist = sdata.distribution(spec.pop("name"), **spec)
    for x in values:
        checks.check(label, "find_chance_that_value", f"chance below {x:g}", quiet(dist.find_chance_that_value, is_below=x))
    for p in chances:
        checks.check(label, "find_cut_off", f"cut-off with {p:g} below", quiet(dist.find_cut_off, with_chance_below=p))

agrees  normal(10, 2) | find_chance_that_value | chance below 4: simplydata 0.00134989803163, R 0.00134989803163; relative difference 6.91e-14 (allowed 1e-12)
      Source: R (run reference) - https://search.r-project.org/R/refmans/stats/html/Normal.html
agrees  normal(10, 2) | find_chance_that_value | chance below 7.5: simplydata 0.105649773667, R 0.105649773667; relative difference 1.44e-15 (allowed 1e-12)
      Source: R (run reference) - https://search.r-project.org/R/refmans/stats/html/Normal.html
agrees  normal(10, 2) | find_chance_that_value | chance below 10: simplydata 0.5, R 0.5; relative difference 0.00e+00 (allowed 1e-12)
      Source: R (run reference) - https://search.r-project.org/R/refmans/stats/html/Normal.html
agrees  normal(10, 2) | find_chance_that_value | chance below 12.3: simplydata 0.874928064363, R 0.874928064363; relative difference 1.27e-16 (allowed 1e-12)
      Source: R (run reference) - https://search.r-project.org/R/refmans/stats/html/Normal.html
agrees  

## Result

In [4]:
checks.finish()

07_distributions: 44 comparisons - 44 agrees
of the SKIPs, 0 are waiting for a simplydata word (see the reason column)


,notebook,dataset,method,quantity,simplydata,reference,reference source,difference,tolerance,tolerance type,score,status,reason,result,finding,explanation,link,kind,url,ran on
0,07_distributions,standard normal table,find_chance_that_value,chance above 1.96,0.024998,0.025000,"NIST/SEMATECH e-Handbook 1.3.6.7, printed",-2.104852e-06,5.000000e-06,absolute,2.104852e-06,PASS,,agrees,,,,printed,https://www.itl.nist.gov/div898/handbook/eda/s...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
1,07_distributions,t table,find_cut_off,"upper 2.5 % point, 10 df",2.228139,2.228000,"NIST/SEMATECH e-Handbook 1.3.6.7, printed",1.388520e-04,5.000000e-04,absolute,1.388520e-04,PASS,,agrees,,,,printed,https://www.itl.nist.gov/div898/handbook/eda/s...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
2,07_distributions,chi-square table,find_cut_off,"upper 5 % point, 10 df",18.307038,18.307000,"NIST/SEMATECH e-Handbook 1.3.6.7, printed",3.805328e-05,5.000000e-04,absolute,3.805328e-05,PASS,,agrees,,,,printed,https://www.itl.nist.gov/div898/handbook/eda/s...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
3,07_distributions,F table,find_cut_off,"upper 5 % point, 5 and 10 df",3.325835,3.326000,"NIST/SEMATECH e-Handbook 1.3.6.7, printed",-1.654696e-04,5.000000e-04,absolute,1.654696e-04,PASS,,agrees,,,,printed,https://www.itl.nist.gov/div898/handbook/eda/s...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
4,07_distributions,"normal(10, 2)",find_chance_that_value,chance below 4,0.001350,0.001350,R,9.324139e-17,1.000000e-12,relative,6.907291e-14,PASS,,agrees,,,,run reference,https://search.r-project.org/R/refmans/stats/h...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
5,07_distributions,"normal(10, 2)",find_chance_that_value,chance below 7.5,0.105650,0.105650,R,1.526557e-16,1.000000e-12,relative,1.444922e-15,PASS,,agrees,,,,run reference,https://search.r-project.org/R/refmans/stats/h...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
6,07_distributions,"normal(10, 2)",find_chance_that_value,chance below 10,0.500000,0.500000,R,0.000000e+00,1.000000e-12,relative,0.000000e+00,PASS,,agrees,,,,run reference,https://search.r-project.org/R/refmans/stats/h...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
7,07_distributions,"normal(10, 2)",find_chance_that_value,chance below 12.3,0.874928,0.874928,R,1.110223e-16,1.000000e-12,relative,1.268931e-16,PASS,,agrees,,,,run reference,https://search.r-project.org/R/refmans/stats/h...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
8,07_distributions,"normal(10, 2)",find_chance_that_value,chance below 17,0.999767,0.999767,R,1.110223e-16,1.000000e-12,relative,1.110481e-16,PASS,,agrees,,,,run reference,https://search.r-project.org/R/refmans/stats/h...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
9,07_distributions,"normal(10, 2)",find_cut_off,cut-off with 0.001 below,3.819535,3.819535,R,0.000000e+00,1.000000e-12,relative,0.000000e+00,PASS,,agrees,,,,run reference,https://search.r-project.org/R/refmans/stats/h...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
